# Layer-wise CDNV (Class-Distance Normalized Variance)

`layer_wise_cdnv_{BCIC2A,TUAB,TUEV}.py`가 slicing_mode/feature_mode 조합별로 저장한 4개의 CSV를 불러와서,

- x축: layer
- y축: `cdnv` (Galanti et al., 2022)
- 한 그래프 안에 6개 method(frozen, linear_probe, finetune, LORA, VERA, dynamic_pearl)를 겹쳐서 표시

형태로 그립니다. `layer_wise_class_separation_plots.ipynb`와 동일한 구조입니다.

- `DATASET`: `'BCIC2A'` / `'TUAB'` / `'TUEV'` 중 선택
- `SPLIT_TO_PLOT`: `'train'` / `'valid'` / `'test'` / `'average'` 중 선택

CDNV는 class separation R^2와 방향이 반대로, **값이 낮을수록** 클래스가 잘 분리된 것입니다. 값의 범위가 몇 자릿수에 걸쳐 있으면 `Y_LOG_SCALE = True`로 바꾸세요.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 0. 설정값
# ==========================================
DATASET = "BCIC2A"  # 'BCIC2A', 'TUAB', 'TUEV'

# 각 layer_wise_cdnv_*.py의 OUTPUT_DIR과 동일
LOG_DIRS = {
    "BCIC2A": "/workspace/EEGPT/dataset/BCIC-2A/layer_wise",
    "TUAB": "/workspace/EEGPT/dataset/TUAB/layer_wise",
    "TUEV": "/workspace/EEGPT/dataset/TUEV/layer_wise",
}
LOG_DIR = LOG_DIRS[DATASET]
CSV_DATE_TAG = "261002"

# 예: 261002_layer_wise_cdnv_raw_flatten.csv
def csv_path(slicing_mode, feature_mode):
    return (
        f"{LOG_DIR}/{CSV_DATE_TAG}_layer_wise_cdnv_"
        f"{slicing_mode}_{feature_mode}.csv"
    )

# (slicing_mode, feature_mode) 4가지 조합
COMBOS = [
    ("raw", "flatten"),
    ("raw", "pooling"),
    ("sliced", "flatten"),
    ("sliced", "pooling"),
]

METRIC = "cdnv"

# 지금은 test set만, 나중에 'train' / 'valid' / 'average'로 바꿔서 재실행하면 됨
SPLIT_TO_PLOT = "test"

# CDNV 값이 여러 자릿수에 걸쳐 있으면 True로 바꿔 로그 스케일로 표시
Y_LOG_SCALE = False

# CSV의 method 값 -> 범례에 표시할 라벨 매핑
METHOD_LABELS = {
    "frozen": "Frozen",
    "linear_probe": "Linear probe",
    "finetune": "Fine tuning",
    "LORA": "LORA",
    "VERA": "VERA",
    "dynamic_pearl": "Dynamic Pearl",
}

METHOD_STYLES = {
    "Frozen":        {"color": "#808080", "linestyle": (0, (1, 1)),         "marker": "x"},
    "Linear probe":  {"color": "#2A78D6", "linestyle": (0, (3, 1, 1, 1)), "marker": "v"},
    "VERA":          {"color": "#EDA100", "linestyle": "--",             "marker": "s"},
    "Pearl":         {"color": "#1BAF7A", "linestyle": "-.",             "marker": "^"},
    "Fine tuning":   {"color": "#4A3AA7", "linestyle": ":",              "marker": "D"},
    "Dynamic Pearl": {"color": "#E34948", "linestyle": "-",              "marker": "o"},
    "LORA":          {"color": "#008300", "linestyle": (0, (5, 1)),      "marker": "P"},
}

In [ ]:
# ==========================================
# 1. CSV 로드
# ==========================================
def load_cdnv_csv(slicing_mode, feature_mode):
    path = csv_path(slicing_mode, feature_mode)
    df = pd.read_csv(path)
    return df

In [ ]:
# ==========================================
# 2. 그래프 함수 (x축=layer, y축=cdnv, method별 선 오버레이)
# ==========================================
def plot_cdnv(df, split, slicing_mode, feature_mode, ax=None):
    sub = df[df["split"] == split]

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    for method_key, label in METHOD_LABELS.items():
        m = sub[sub["method"] == method_key].sort_values("layer")
        if m.empty:
            continue
        style = METHOD_STYLES.get(label, {})
        ax.plot(m["layer"], m[METRIC], label=label, **style)

    if Y_LOG_SCALE:
        ax.set_yscale("log")

    ax.set_xlabel("Layer")
    ax.set_ylabel("CDNV (lower = better separation)")
    ax.set_title(f"CDNV - {DATASET}\n({slicing_mode}+{feature_mode}, {split} set)")
    ax.grid(alpha=0.3)
    ax.legend()
    return ax

In [ ]:
# ==========================================
# 3. 4개 조합 전체 그래프 그리기
# ==========================================
for slicing_mode, feature_mode in COMBOS:
    df = load_cdnv_csv(slicing_mode, feature_mode)

    plot_cdnv(df, SPLIT_TO_PLOT, slicing_mode, feature_mode)
    plt.tight_layout()
    plt.show()